In [1]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import norm

#### **Parte 1. Martingalas en tiempo discreto**
##### **Caminata aleatoria: Y_n = X_n² − n**
X_n es una caminata aleatoria simétrica con incrementos ±1 con probabilidad 1/2. El proceso Y_n = X_n² − n es una martingala con E[Y_n] = 0.

In [ ]:
np.random.seed(123)

# Parámetros
n_simulaciones = 50
T = 100
n_vals = np.arange(T + 1)

# Caminata aleatoria simétrica
incrementos = np.random.choice([-1, 1], size=(n_simulaciones, T))
X = np.hstack((np.zeros((n_simulaciones, 1)), np.cumsum(incrementos, axis=1)))

# Proceso Y_n = X_n^2 - n
Y = X**2 - n_vals

plt.figure(figsize=(10, 6))
for i in range(n_simulaciones):
    plt.plot(n_vals, Y[i, :], alpha=0.6)
plt.axhline(0, color='black', linestyle='--', linewidth=2, label='Esperanza Teórica = 0')
plt.title("50 trayectorias de $Y_n = X_n^2 - n$")
plt.xlabel("n")
plt.ylabel("$Y_n$")
plt.legend()
plt.show()

for n in [20, 50, 100]:
    print(f"E(Y_{n}) ≈ {np.mean(Y[:, n]):.4f}")

##### **Transformada de martingala**

Y_n = Σ φ_k ξ_k, donde ξ_k = ±1 con probabilidad 1/2 y φ_k es una estrategia predecible: si la apuesta anterior se gana se apuesta 1, y si se pierde se apuesta lo necesario para regresar a 1. Como φ_k es predecible, Y_n es martingala con E[Y_n] = 0. La media simulada sale positiva porque la estrategia gana poco casi siempre y pierde mucho en raras ocasiones, que con 50 trayectorias casi no aparecen.

In [ ]:
np.random.seed(123)

n_simulaciones = 50
T = 100
Y = np.zeros((n_simulaciones, T + 1))

for i in range(n_simulaciones):
    xi = np.random.choice([-1, 1], size=T)
    phi = np.zeros(T + 1)
    phi[1] = 1

    for k in range(1, T + 1):
        Y[i, k] = Y[i, k-1] + phi[k] * xi[k-1]

        # Estrategia: si gana apuesta 1; si pierde, apuesta lo necesario para quedar en 1
        if k < T:
            if xi[k-1] == 1:
                phi[k+1] = 1
            else:
                phi[k+1] = -Y[i, k] + 1

plt.figure(figsize=(10, 6))
for i in range(n_simulaciones):
    plt.plot(Y[i, :], alpha=0.6)
plt.axhline(0, color='black', linestyle='--', linewidth=2, label='Esperanza Teórica = 0')
plt.title("50 trayectorias de la transformada de martingala $Y_n$")
plt.xlabel("n")
plt.ylabel("$Y_n$")
plt.legend()
plt.show()

for n in [20, 50, 100]:
    print(f"E(Y_{n}) ≈ {np.mean(Y[:, n]):.4f}")

##### **Caminata asimétrica: martingala de De Moivre**

X_n es una caminata con P(ξ = 1) = p = 0.55 y P(ξ = −1) = 1 − p. El proceso Y_n = ((1 − p)/p)^X_n es una martingala con E[Y_n] = 1.

In [ ]:
np.random.seed(123)

# Parámetros
p = 0.55
n_simulaciones = 50
T = 100
c = (1 - p) / p

# Caminata asimétrica: P(ξ = 1) = p, P(ξ = -1) = 1 - p
xi = np.random.choice([1, -1], size=(n_simulaciones, T), p=[p, 1 - p])
X = np.hstack((np.zeros((n_simulaciones, 1)), np.cumsum(xi, axis=1)))

# Martingala de De Moivre: Y_n = ((1-p)/p)^X_n
Y = c ** X

plt.figure(figsize=(10, 6))
for i in range(n_simulaciones):
    plt.plot(Y[i, :], alpha=0.6)
plt.axhline(1, color='black', linestyle='--', linewidth=2, label='Esperanza Teórica = 1')
plt.title(f"50 trayectorias de $Y_n = ((1-p)/p)^{{X_n}}$ con p = {p}")
plt.xlabel("n")
plt.ylabel("$Y_n$")
plt.legend()
plt.show()

for n in [20, 50, 100]:
    print(f"E(Y_{n}) ≈ {np.mean(Y[:, n]):.4f}")

#### **Parte 2. Martingalas del proceso de Poisson**

##### **Martingalas de un proceso de Poisson con λ = 5**

Si N_t es un proceso de Poisson con intensidad λ, entonces X_t = (N_t − λt)² − λt es martingala con E[X_t] = 0, e Y_t = exp(N_t ln(1 − u) + uλt) es martingala con E[Y_t] = 1 (con u = 0.5).

In [ ]:
lam = 5
T = 10
num_trayectorias = 20
puntos = 1000
t = np.linspace(0, T, puntos)
dt = t[1] - t[0]

plt.figure(figsize=(10, 6))
np.random.seed(42)
for _ in range(num_trayectorias):
    incrementos = np.random.poisson(lam * dt, size=puntos - 1)
    Nt = np.concatenate(([0], np.cumsum(incrementos)))
    Xt = (Nt - lam * t)**2 - lam * t
    plt.plot(t, Xt, lw=1.5, alpha=0.6)

plt.axhline(0, color='black', linestyle='--', linewidth=2, label='Esperanza E[X_t] = 0')
plt.title(r'20 trayectorias de $X_t = (N_t - \lambda t)^2 - \lambda t$', fontsize=14)
plt.xlabel('Tiempo (t)', fontsize=12)
plt.ylabel(r'$X_t$', fontsize=12)
plt.legend()
plt.grid(False)
plt.tight_layout()
plt.show()

In [ ]:
lam = 5
T = 10
u = 0.5
num_trayectorias = 20
puntos = 1000
t = np.linspace(0, T, puntos)
dt = t[1] - t[0]

plt.figure(figsize=(10, 6))
np.random.seed(10)
for _ in range(num_trayectorias):
    incrementos = np.random.poisson(lam * dt, size=puntos - 1)
    Nt = np.concatenate(([0], np.cumsum(incrementos)))
    Yt = np.exp(Nt * np.log(1 - u) + u * lam * t)
    plt.plot(t, Yt, lw=1.5, alpha=0.6)

plt.axhline(1, color='black', linestyle='--', linewidth=2, label='Esperanza E[Y_t] = 1')
plt.title(r'20 trayectorias de $Y_t = e^{N_t \ln(1-u) + u\lambda t}$', fontsize=14)
plt.xlabel('Tiempo (t)', fontsize=12)
plt.ylabel(r'$Y_t$', fontsize=12)
plt.legend()
plt.grid(False)
plt.tight_layout()
plt.show()

##### **Poisson compensado y martingala exponencial con λ = 2**

X_t = N_t − λt (Poisson compensado) es martingala con E[X_t] = 0, e Y_t = exp(N_t − λt(e − 1)) es martingala con E[Y_t] = 1.

La media simulada de Y_t sale prácticamente en 0 y no es un error: Y_t tiende a 0 casi seguramente, pero mantiene E[Y_t] = 1 gracias a trayectorias rarísimas que alcanzan valores enormes. Con 30 simulaciones no aparece ninguna.

In [ ]:
lam = 2
num_trayectorias = 30
T_plot = 20
puntos = 1000
t = np.linspace(0, T_plot, puntos)
dt = t[1] - t[0]

plt.figure(figsize=(10, 6))
np.random.seed(42)
for _ in range(num_trayectorias):
    incrementos = np.random.poisson(lam * dt, size=puntos - 1)
    Nt = np.concatenate(([0], np.cumsum(incrementos)))
    Xt = Nt - lam * t
    plt.plot(t, Xt, lw=1.5, alpha=0.6)

plt.axhline(0, color='black', linestyle='--', linewidth=2, label='Esperanza Teórica = 0')
plt.title(r'30 trayectorias de $X_t = N_t - \lambda t$ (Poisson Compensado)', fontsize=14)
plt.xlabel('Tiempo (t)')
plt.ylabel(r'$X_t$')
plt.legend()
plt.grid(False)
plt.tight_layout()
plt.show()

for t_val in [20, 50]:
    N_t_sim = np.random.poisson(lam * t_val, num_trayectorias)
    X_t_sim = N_t_sim - lam * t_val
    print(f"Aproximación de E(X_{t_val}) con 30 simulaciones: {np.mean(X_t_sim):.4f} (Teórico: 0)")

In [ ]:
lam = 2
num_trayectorias = 30
T_plot = 20
puntos = 1000
t = np.linspace(0, T_plot, puntos)
dt = t[1] - t[0]

plt.figure(figsize=(10, 6))
np.random.seed(15)
for _ in range(num_trayectorias):
    incrementos = np.random.poisson(lam * dt, size=puntos - 1)
    Nt = np.concatenate(([0], np.cumsum(incrementos)))
    Yt = np.exp(Nt - lam * t * (np.e - 1))
    plt.plot(t, Yt, lw=1.5, alpha=0.6)

plt.axhline(1, color='black', linestyle='--', linewidth=2, label='Esperanza Teórica = 1')
plt.title(r'30 trayectorias de $Y_t = e^{N_t - \lambda t(e-1)}$', fontsize=14)
plt.xlabel('Tiempo (t)')
plt.ylabel(r'$Y_t$')
plt.legend()
plt.grid(False)
plt.tight_layout()
plt.show()

for t_val in [20, 50]:
    N_t_sim = np.random.poisson(lam * t_val, num_trayectorias)
    Y_t_sim = np.exp(N_t_sim - lam * t_val * (np.e - 1))
    print(f"Aproximación de E(Y_{t_val}) con 30 simulaciones: {np.mean(Y_t_sim):.4f} (Teórico: 1)")

##### **Proceso de Poisson compuesto**

X_t = Y_1 + … + Y_{N_t}, con N_t Poisson de intensidad λ = 2 y saltos Y_i ~ N(0, 4). Como E[Y_i] = 0, X_t es martingala con E[X_t] = 0.

In [ ]:
# Parámetros del proceso de Poisson
lam = 2
T = 20
num_trayectorias = 30

# Saltos Y_i ~ N(0, 4)
mu_y = 0
sigma_y = 2

plt.figure(figsize=(10, 6))
np.random.seed(42)
for _ in range(num_trayectorias):
    N_T = np.random.poisson(lam * T)
    if N_T > 0:
        tiempos_salto = np.sort(np.random.uniform(0, T, N_T))
        saltos = np.random.normal(mu_y, sigma_y, N_T)
        tiempos = np.concatenate(([0], tiempos_salto, [T]))
        valores_X = np.cumsum(saltos)
        valores = np.concatenate(([0], valores_X, [valores_X[-1]]))
    else:
        tiempos = [0, T]
        valores = [0, 0]
    plt.step(tiempos, valores, where='post', lw=1.5, alpha=0.5)

plt.axhline(0, color='black', linestyle='--', linewidth=2, label='Esperanza Teórica E[X_t] = 0')
plt.title(r'30 trayectorias del Proceso Poisson Compuesto ($Y_i \sim \mathcal{N}(0, 4)$)', fontsize=14)
plt.xlabel('Tiempo (t)', fontsize=12)
plt.ylabel(r'$X_t$', fontsize=12)
plt.legend()
plt.tight_layout()
plt.show()

##### **Proceso de Poisson compuesto compensado**

X_t es Poisson compuesto con λ = 20 y saltos Y_i ~ Gamma(5, 2). El proceso compensado Z_t = X_t − λt·E[Y] es martingala con E[Z_t] = 0.

In [ ]:
lam = 20
T_plot = 20
num_trayectorias = 30

# Saltos Y_i ~ Gamma(5, 2)
alpha_y = 5
scale_y = 2
E_Y = alpha_y * scale_y
deriva = lam * E_Y

plt.figure(figsize=(12, 6))
np.random.seed(42)
for _ in range(num_trayectorias):
    N_T = np.random.poisson(lam * T_plot)
    if N_T > 0:
        tiempos_salto = np.sort(np.random.uniform(0, T_plot, N_T))
        saltos = np.random.gamma(shape=alpha_y, scale=scale_y, size=N_T)
        t_plot = [0]
        z_plot = [0]
        X_acumulado = 0
        for t_i, y_i in zip(tiempos_salto, saltos):
            t_plot.append(t_i)
            z_plot.append(X_acumulado - deriva * t_i)
            X_acumulado += y_i
            t_plot.append(t_i)
            z_plot.append(X_acumulado - deriva * t_i)
        t_plot.append(T_plot)
        z_plot.append(X_acumulado - deriva * T_plot)
    else:
        t_plot = [0, T_plot]
        z_plot = [0, -deriva * T_plot]
    plt.plot(t_plot, z_plot, lw=1.2, alpha=0.6)

plt.axhline(0, color='black', linestyle='--', linewidth=2, label=r'Esperanza Teórica $E[Z_t] = 0$')
plt.title(r'30 trayectorias de $Z_t = X_t - \lambda t E[Y]$ (Poisson Compuesto)', fontsize=14)
plt.xlabel('Tiempo (t)', fontsize=12)
plt.ylabel(r'$Z_t$', fontsize=12)
plt.legend()
plt.grid(False)
plt.tight_layout()
plt.show()

print("Aproximaciones de la Esperanza (Teórico: 0)")
for t_val in [20, 50]:
    N_t_sim = np.random.poisson(lam * t_val, num_trayectorias)
    Z_t_sim = [np.sum(np.random.gamma(shape=alpha_y, scale=scale_y, size=n)) - deriva * t_val
               for n in N_t_sim]
    print(f"E[Z_{t_val}] usando 30 simulaciones: {np.mean(Z_t_sim):.4f}")

#### **Parte 3. Movimiento browniano**

##### **Martingala exponencial: X_t = exp(cB_t − c²t/2)**

Con c = 2, X_t es martingala con E[X_t] = 1. Igual que en el caso de Poisson, las trayectorias tienden a 0 y la media empírica cae por debajo de 1, aunque la esperanza teórica se mantiene en 1. Este proceso es la base del movimiento browniano geométrico con el que se modelan los precios de las acciones en Black-Scholes.

In [ ]:
c = 2
Tmax = 20
n_sim = 50
n_steps = 1000
dt = Tmax / n_steps
tiempos = np.linspace(0, Tmax, n_steps + 1)

np.random.seed(123)
incrementos = np.random.normal(0, np.sqrt(dt), size=(n_steps, n_sim))
B = np.vstack([np.zeros(n_sim), np.cumsum(incrementos, axis=0)])
X_mat = np.exp(c * B - 0.5 * c**2 * tiempos[:, np.newaxis])
media_empirica = np.mean(X_mat, axis=1)

plt.figure(figsize=(12, 6))
colores_vivos = ['#FF3F3F', '#3FFF3F', '#3F3FFF', '#FF3FFF', '#3FFFFF',
                 '#FF7F00', '#7F00FF', '#00FF7F', '#FF007F', '#FFD700']
for j in range(n_sim):
    plt.plot(tiempos, X_mat[:, j], color=colores_vivos[j % 10], alpha=0.6, lw=1.2)

plt.plot(tiempos, media_empirica, color='black', lw=2.5, label='Media Empírica', zorder=100)
plt.axhline(1, color='red', linestyle='--', lw=2.5, label='Esperanza Teórica (1)', zorder=101)
plt.ylim(-0.5, 10)
plt.title(r"Simulación de $X_t = \exp(c B_t - \frac{1}{2} c^2 t)$", fontsize=14, pad=25)
plt.xlabel("Tiempo (t)", fontsize=12)
plt.ylabel("$X_t$", fontsize=12)
plt.legend(loc='upper right', frameon=True, shadow=True)
plt.tight_layout()
plt.show()

##### **Martingalas del browniano: B_t² − t y exp(B_t − t/2)**

B_t² − t es martingala con esperanza 0 y exp(B_t − t/2) es martingala con esperanza 1. Se simulan 30 trayectorias y se aproximan las esperanzas en t = 20 y t = 50.

In [ ]:
np.random.seed(42)
N = 30
t_max = 50
pasos = 5000
dt = t_max / pasos
t = np.linspace(0, t_max, pasos + 1)

incrementos = np.random.normal(0, np.sqrt(dt), size=(N, pasos))
B = np.hstack((np.zeros((N, 1)), np.cumsum(incrementos, axis=1)))

# a) B_t^2 - t
X = B**2 - t

# b) exp(B_t - t/2)
Y = np.exp(B - t/2)

idx_20 = int((20 / t_max) * pasos)
idx_50 = pasos

print("Inciso a)")
print(f"E[B_t^2 - t] en t=20: {np.mean(X[:, idx_20]):.4f} (Teórico: 0)")
print(f"E[B_t^2 - t] en t=50: {np.mean(X[:, idx_50]):.4f} (Teórico: 0)\n")
print("Inciso b)")
print(f"E[exp(B_t - t/2)] en t=20: {np.mean(Y[:, idx_20]):.4f} (Teórico: 1)")
print(f"E[exp(B_t - t/2)] en t=50: {np.mean(Y[:, idx_50]):.4f} (Teórico: 1)")

t_20 = t[:idx_20 + 1]
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.plot(t_20, X[:, :idx_20 + 1].T, alpha=0.5, color='purple')
plt.axhline(0, color='black', linewidth=2, linestyle='--', label='Esperanza (0)')
plt.title('30 Trayectorias de $B_t^2 - t$')
plt.xlabel('Tiempo t')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(t_20, Y[:, :idx_20 + 1].T, alpha=0.5, color='orange')
plt.axhline(1, color='black', linewidth=2, linestyle='--', label='Esperanza (1)')
plt.title('30 Trayectorias de $e^{B_t - t/2}$')
plt.xlabel('Tiempo t')
plt.legend()

plt.tight_layout()
plt.show()

##### **Propiedades del browniano e integral de Itô**

Con 10,000 trayectorias se verifican por simulación: P(B_1 ≤ 0, B_2 ≤ 0) = 3/8; P(∫₀¹ B_t dt > 2/√3), usando que la integral se distribuye N(0, 1/3); la variación cuadrática en [0, 1], que debe ser 1; y la integral de Itô ∫₀¹⁰ B_s dB_s = ½B₁₀² − 5, con esperanza 0.

In [ ]:
np.random.seed(42)
N = 10000    # Número de trayectorias
pasos = 1000 # Discretización

# a) P(B_1 <= 0, B_2 <= 0)
B1 = np.random.normal(0, 1, N)
B2 = B1 + np.random.normal(0, 1, N)
prob_a_sim = np.mean((B1 <= 0) & (B2 <= 0))
print("a) P(B_1 <= 0, B_2 <= 0)")
print(f"   Simulación: {prob_a_sim:.4f}")
print(f"   Exacta:     {3/8:.4f}")

# b) P(∫_0^1 B_t dt > 2/√3)
dt_b = 1.0 / pasos
incrementos_b = np.random.normal(0, np.sqrt(dt_b), (N, pasos))
B_trayectorias_1 = np.hstack((np.zeros((N, 1)), np.cumsum(incrementos_b, axis=1)))
integral_B = np.sum(B_trayectorias_1[:, :-1] * dt_b, axis=1)
print("b) P(∫ B_t dt > 2/√3)")
print(f"   Simulación: {np.mean(integral_B > 2/np.sqrt(3)):.4f}")
print(f"   Exacta:     {1 - norm.cdf(2):.4f}")

# d) Variación cuadrática en [0, 1]
variacion_cuadratica = np.sum(np.diff(B_trayectorias_1, axis=1)**2, axis=1)
print("d) Variación cuadrática en t = 1")
print(f"   Simulación: {np.mean(variacion_cuadratica):.4f}")
print(f"   Teórica:    1.0000")

# e) Integral de Itô ∫_0^10 B_s dB_s
t_10 = 10.0
dt_e = t_10 / pasos
incrementos_e = np.random.normal(0, np.sqrt(dt_e), (N, pasos))
B_trayectorias_10 = np.hstack((np.zeros((N, 1)), np.cumsum(incrementos_e, axis=1)))
integral_ito = np.sum(B_trayectorias_10[:, :-1] * incrementos_e, axis=1)
formula = 0.5 * B_trayectorias_10[:, -1]**2 - 0.5 * t_10
print("e) Integral de Itô ∫ B_s dB_s en [0, 10]")
print(f"   Diferencia media con (1/2)B_10^2 - 5: {np.mean(np.abs(integral_ito - formula)):.4f}")
print(f"   Esperanza simulada:                   {np.mean(integral_ito):.4f}")
print(f"   Esperanza teórica:                    0.0000")

#### **Martingala Y_t = tB_t − ∫₀ᵗ B_s ds**

Y_t = tB_t − ∫₀ᵗ B_s ds es martingala con E[Y_t] = 0 y Var(Y_t) = t³/3. En t = 20 la desviación estándar es ≈ 51.6, así que con 30 trayectorias la media simulada puede alejarse varias unidades de 0.

In [ ]:
np.random.seed(42)
N = 30
t_max = 20.0
pasos = 2000
dt = t_max / pasos
t = np.linspace(0, t_max, pasos + 1)

incrementos = np.random.normal(0, np.sqrt(dt), size=(N, pasos))
B = np.hstack((np.zeros((N, 1)), np.cumsum(incrementos, axis=1)))

integral_B = np.zeros_like(B)
integral_B[:, 1:] = np.cumsum(B[:, :-1] * dt, axis=1)

Y = t * B - integral_B

print(f"Resultados en t = {t_max}:")
print(f"Esperanza simulada E[Y_20]: {np.mean(Y[:, -1]):.4f}")
print(f"Valor exacto teórico:       0.0000")

plt.figure(figsize=(10, 6))
plt.plot(t, Y.T, alpha=0.5, color="#c33654")
plt.axhline(0, color='#e74c3c', linewidth=2, linestyle='--', label='Esperanza Exacta (0)')
plt.plot(t, np.mean(Y, axis=0), color='#f1c40f', linewidth=3, label='Media Empírica')
plt.title(r'30 Trayectorias de $Y_t = tB_t - \int_0^t B_s ds$')
plt.xlabel('Tiempo t')
plt.ylabel('$Y_t$')
plt.legend()
plt.tight_layout()
plt.show()